# Pular IA — Génération du Dataset LLM

Ce notebook tire les données **directement depuis ton serveur Railway** (pas d'upload manuel),
génère le dataset, et sauvegarde sur Google Drive pour le fine-tuning.

**Flux :**
```
Railway (corpus communautaire) → Colab (génère dataset) → Drive (sauvegarde) → Fine-tuning
```

**Durée :** ~2 min | **GPU requis :** Non

## Config — À remplir UNE FOIS

In [ ]:
# ── Remplis ces deux variables ────────────────────────────────────────────────
WEBAPP_URL = "https://TON-APP.railway.app"  # URL de ton app Railway (sans slash final)
ADMIN_KEY  = ""                              # Valeur de ADMIN_KEY dans .env Railway (laisser vide si non défini)

# ── Google Drive ──────────────────────────────────────────────────────────────
DRIVE_ROOT = "/content/drive/MyDrive/pular-ia"

# ── Chemins locaux Colab ──────────────────────────────────────────────────────
CORPUS_LOCAL  = "/content/corpus-pular"
SCRIPTS_LOCAL = "/content/scripts"
DATASET_LOCAL = f"{CORPUS_LOCAL}/dataset/llm"

# Vérification
if "TON-APP" in WEBAPP_URL:
    print("ATTENTION : Change WEBAPP_URL par l'URL de ton app Railway !")
else:
    print(f"Config OK — Serveur : {WEBAPP_URL}")

## Étape 1 — Monter Google Drive + installer les outils

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os, shutil, json, zipfile, io, requests

# Créer les dossiers Drive
for d in [
    f"{DRIVE_ROOT}/datasets/llm",
    f"{DRIVE_ROOT}/scripts",
]:
    os.makedirs(d, exist_ok=True)

print("Drive monté ✓")

## Étape 2 — Vérifier l'état du corpus sur le serveur

In [ ]:
# Affiche les stats du corpus sur Railway avant de télécharger
import requests, json

params = {"key": ADMIN_KEY} if ADMIN_KEY else {}

try:
    r = requests.get(f"{WEBAPP_URL}/api/admin/stats-corpus", params=params, timeout=15)
    r.raise_for_status()
    stats = r.json()

    print("=== Corpus sur le serveur ===")
    print(f"  Contributions       : {stats['contributions']}")
    for status, n in stats.get('contributions_status', {}).items():
        print(f"    └─ {status:<12} : {n}")
    print(f"  Corrections         : {stats['corrections']}")
    print(f"  Transcriptions      : {stats['transcriptions']}")
    print(f"  Mots custom profs   : {stats['mots_custom']}")
    print(f"  Chunks RAG          : {stats['rag_chunks']} ({stats['rag_livres']} livres)")
    print(f"  Dernière MAJ        : {stats['timestamp'][:19]}")

except requests.exceptions.ConnectionError:
    print("ERREUR : Impossible de joindre le serveur.")
    print(f"  Vérifie que {WEBAPP_URL} est bien démarré sur Railway.")
except Exception as e:
    print(f"ERREUR : {e}")

## Étape 3 — Télécharger le corpus depuis Railway

In [ ]:
import requests, zipfile, io, os, shutil

print("Téléchargement du corpus depuis Railway...")
params = {"key": ADMIN_KEY} if ADMIN_KEY else {}

r = requests.get(
    f"{WEBAPP_URL}/api/admin/export-corpus",
    params=params,
    timeout=120,  # peut prendre du temps si gros corpus
    stream=True,
)
r.raise_for_status()

contenu = b""
for chunk in r.iter_content(chunk_size=8192):
    contenu += chunk

print(f"ZIP reçu : {len(contenu) / 1024:.0f} KB")

# Extraire dans /content/
if os.path.exists(CORPUS_LOCAL):
    shutil.rmtree(CORPUS_LOCAL)

with zipfile.ZipFile(io.BytesIO(contenu)) as z:
    z.extractall("/content/")

# Compter ce qu'on a reçu
for dossier in [
    "community/contributions",
    "community/corrections",
    "processed/transcriptions",
    "dataset/translit",
    "jeu",
]:
    chemin = f"{CORPUS_LOCAL}/{dossier}"
    if os.path.exists(chemin):
        n = len(os.listdir(chemin))
        print(f"  {dossier}: {n} fichiers")

print("\nCorpus extrait dans /content/corpus-pular/ ✓")

## Étape 4 — Charger le script de génération

In [ ]:
import os, shutil

os.makedirs(SCRIPTS_LOCAL, exist_ok=True)

# Option A : depuis Drive (si tu y as déjà copié les scripts)
scripts_drive = f"{DRIVE_ROOT}/scripts"
scripts_charges = False

if os.path.exists(f"{scripts_drive}/prepare_llm_dataset.py"):
    shutil.copy(f"{scripts_drive}/prepare_llm_dataset.py", f"{SCRIPTS_LOCAL}/prepare_llm_dataset.py")
    scripts_charges = True
    print("Script chargé depuis Drive ✓")

if not scripts_charges:
    # Option B : uploader directement
    print("Script non trouvé sur Drive.")
    print("Upload prepare_llm_dataset.py ci-dessous (une seule fois, il sera sauvé sur Drive) :")
    from google.colab import files
    uploaded = files.upload()
    for nom, contenu in uploaded.items():
        with open(f"{SCRIPTS_LOCAL}/{nom}", 'wb') as f:
            f.write(contenu)
        # Sauver sur Drive pour les prochaines sessions
        with open(f"{scripts_drive}/{nom}", 'wb') as f:
            f.write(contenu)
        print(f"  {nom} chargé et sauvé sur Drive ✓")

## Étape 5 — Générer le dataset

In [ ]:
# Option A : Génération locale dans Colab (rapide, résultat visible ici)
!python {SCRIPTS_LOCAL}/prepare_llm_dataset.py --root /content --min-qualite bronze --seed 42

In [ ]:
# Option B : Génération sur le serveur Railway (le dataset reste sur le serveur)
# Utile si tu veux juste déclencher une mise à jour sans télécharger le corpus

import requests
params = {"key": ADMIN_KEY} if ADMIN_KEY else {}

print("Génération du dataset sur Railway...")
r = requests.post(f"{WEBAPP_URL}/api/admin/generer-dataset", params=params, timeout=180)
r.raise_for_status()
result = r.json()

stats = result.get("stats", {})
print(f"Dataset généré sur Railway :")
print(f"  Total   : {stats.get('total', 0)} exemples")
print(f"  Train   : {stats.get('train', 0)}")
print(f"  Val     : {stats.get('val', 0)}")
print(f"  Test    : {stats.get('test', 0)}")

## Étape 6 — Aperçu du dataset

In [ ]:
import json

train_path = f"{DATASET_LOCAL}/train.jsonl"

with open(train_path, encoding="utf-8") as f:
    lignes = f.readlines()

print(f"Total train : {len(lignes)} exemples\n")

# Afficher un exemple de chaque source
sources_vues = set()
for ligne in lignes:
    ex = json.loads(ligne)
    src = ex.get('source', '?')
    if src not in sources_vues:
        sources_vues.add(src)
        print(f"--- {src} [{ex.get('qualite','?')}] ---")
        print(f"  INSTRUCTION : {ex['instruction']}")
        if ex.get('input'):
            print(f"  INPUT       : {ex['input'][:80]}")
        print(f"  OUTPUT      : {ex['output'][:100]}")
        print()

## Étape 7 — Sauvegarder sur Google Drive

In [ ]:
import shutil, os

DST_DRIVE = f"{DRIVE_ROOT}/datasets/llm"
os.makedirs(DST_DRIVE, exist_ok=True)

for fichier in ["train.jsonl", "val.jsonl", "test.jsonl", "stats.json"]:
    src = f"{DATASET_LOCAL}/{fichier}"
    if os.path.exists(src):
        shutil.copy(src, f"{DST_DRIVE}/{fichier}")
        taille = os.path.getsize(src) / 1024
        print(f"  {fichier:<20} {taille:>6.1f} KB  -> Drive ✓")

print(f"\nDataset sauvegardé : {DST_DRIVE}")
print("Ouvre colab_finetune_mt5.ipynb pour lancer le fine-tuning !")

## Étape 8 — (Optionnel) Ajouter manuellement des données en direct

Tu peux enrichir le dataset ici directement, sans passer par l'interface web.

In [ ]:
import json, uuid
from datetime import datetime
from pathlib import Path

# ── Ajoute tes données ici ────────────────────────────────────────────────────

NOUVEAUX_MOTS = [
    # {"fr": "Bonjour", "pular": "Jam waali", "adlam": "", "cat": "Expressions", "emoji": "👋"},
]

NOUVELLES_PHRASES = [
    # Phrases pular validées que tu connais
    # "Mi yiɗi ɓiɓɓe am",
]

# ── Injecter dans le corpus local ─────────────────────────────────────────────
contrib_dir      = Path(f"{CORPUS_LOCAL}/community/contributions")
mots_custom_path = Path(f"{CORPUS_LOCAL}/jeu/mots_custom.json")

contrib_dir.mkdir(parents=True, exist_ok=True)
mots_custom_path.parent.mkdir(parents=True, exist_ok=True)

# Mots
mots_existants = json.loads(mots_custom_path.read_text()) if mots_custom_path.exists() else []
for mot in NOUVEAUX_MOTS:
    mot.update({"id": str(uuid.uuid4())[:8], "pseudo": "colab", "date": datetime.now().isoformat()})
    mot.setdefault("cat", "Autre")
    mot.setdefault("emoji", "?")
    mot.setdefault("note", "")
    mots_existants.append(mot)
mots_custom_path.write_text(json.dumps(mots_existants, ensure_ascii=False, indent=2))

# Phrases
for phrase in NOUVELLES_PHRASES:
    cid   = str(uuid.uuid4())[:8]
    entry = {"id": cid, "pseudo": "colab", "transcription_auto": "",
             "texte_final": phrase.strip(), "audio": "",
             "timestamp": datetime.now().isoformat(), "source": "colab_manuel", "status": "valider"}
    (contrib_dir / f"{cid}.json").write_text(json.dumps(entry, ensure_ascii=False, indent=2))

print(f"{len(NOUVEAUX_MOTS)} mots ajoutés | {len(NOUVELLES_PHRASES)} phrases ajoutées")
print("Relance l'Etape 5 pour régénérer le dataset avec ces nouvelles données.")

## Étape 9 — Pousser les nouvelles données vers Railway (feedback loop)

Si tu as ajouté des mots/phrases en Étape 8, tu peux les remonter sur le serveur via les endpoints API.

In [ ]:
import requests, json
from pathlib import Path

# Lire les mots custom locaux et les pousser vers Railway
mots_custom_path = Path(f"{CORPUS_LOCAL}/jeu/mots_custom.json")
if not mots_custom_path.exists():
    print("Pas de mots custom à pousser.")
else:
    mots = json.loads(mots_custom_path.read_text())
    # Récupérer les mots déjà sur le serveur pour éviter les doublons
    params = {"key": ADMIN_KEY} if ADMIN_KEY else {}
    existants = requests.get(f"{WEBAPP_URL}/api/prof/mots", params=params).json()
    ids_existants = {m["id"] for m in existants}

    nb_pousses = 0
    for mot in mots:
        if mot["id"] in ids_existants:
            continue  # déjà sur le serveur
        r = requests.post(
            f"{WEBAPP_URL}/api/prof/mot",
            data={
                "emoji":  mot.get("emoji", "?"),
                "fr":     mot["fr"],
                "pular":  mot["pular"],
                "adlam":  mot.get("adlam", ""),
                "cat":    mot.get("cat", "Autre"),
                "note":   mot.get("note", ""),
                "pseudo": mot.get("pseudo", "colab"),
            }
        )
        if r.status_code == 200:
            nb_pousses += 1

    print(f"{nb_pousses} nouveaux mots poussés vers Railway ✓")
    print("Ils apparaîtront dans le jeu au prochain chargement.")

---
## Workflow complet (à répéter à chaque cycle d'enrichissement)

```
1. Communauté contribue via Railway (24h/7j)
2. Prof valide depuis l'Espace Créateur sur Railway
3. Ouvre ce notebook dans Colab
4. Étape 2 → vérifie les stats
5. Étape 3 → télécharge le corpus
6. Étape 5 → génère le dataset
7. Étape 7 → sauvegarde sur Drive
8. Ouvre colab_finetune_mt5.ipynb → fine-tune
```

**Cycle recommandé :** toutes les semaines ou dès que tu as 50+ nouvelles contributions.